## What is Logistic Regression?

Despite the name, I use Logistic Regression for **classification**, not regression — specifically for predicting probabilities that map to discrete classes (most commonly binary: 0 or 1, yes or no, spam or not spam).

The core problem with using plain Linear Regression for classification is that its output is unbounded — it can predict values like -50 or +300, which don't make sense as probabilities. I need something that squashes any real number into the range [0, 1]. That's exactly what the **sigmoid function** does.

## The Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

where $z = w \cdot X + b$ (the same linear combination I use in linear regression).

- As $z \to +\infty$, $\sigma(z) \to 1$
- As $z \to -\infty$, $\sigma(z) \to 0$
- At $z = 0$, $\sigma(z) = 0.5$

So my full hypothesis becomes:

$$\hat{y} = \sigma(w \cdot X + b) = \frac{1}{1 + e^{-(w \cdot X + b)}}$$

This output is interpreted as the **probability** that the sample belongs to class 1. I then apply a threshold (typically 0.5) to convert that probability into a hard class prediction:

$$\text{class} = \begin{cases} 1 & \text{if } \hat{y} \geq 0.5 \\ 0 & \text{if } \hat{y} < 0.5 \end{cases}$$

## Why Not Use MSE as the Cost Function?

If I plugged the sigmoid output into the same squared-error cost function I used for Linear Regression, the resulting cost surface would be **non-convex** — full of local minima — because the sigmoid is nonlinear. Gradient Descent could get stuck rather than reliably finding the global minimum.

Instead, I use **Log Loss** (also called Binary Cross-Entropy), which stays convex for this problem:

$$J(w,b) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

Intuition behind this formula: 
- If the true label $y_i = 1$, the loss reduces to $-\log(\hat{y}_i)$ — this penalizes the model heavily when it predicted a probability close to 0 for something that was actually class 1.
- If $y_i = 0$, the loss reduces to $-\log(1-\hat{y}_i)$ — heavily penalizes confident wrong predictions the other way.

This cost function gets minimized via Gradient Descent, using the same update-rule structure as before — compute gradients of $J$ with respect to $w$ and $b$, then step in the negative gradient direction.

## The Decision Boundary

The line (or in higher dimensions, hyperplane) where $\hat{y} = 0.5$, i.e. where $w \cdot X + b = 0$, is called the **decision boundary**. For simple 2D data, this is a straight line separating the two classes. For more complex data, polynomial features can create curved boundaries, same as with Polynomial Regression.

---

## Classification Metrics

Once I've trained the model, I need different tools than MAE/RMSE/R² to judge performance — those are for continuous outputs. For classification, everything starts from the **Confusion Matrix**.

### Confusion Matrix

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

- **TP**: correctly predicted class 1
- **TN**: correctly predicted class 0
- **FP**: predicted 1, actually 0 (a "false alarm" / Type I error)
- **FN**: predicted 0, actually 1 (a "miss" / Type II error)

### Accuracy

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

- Fraction of all predictions that were correct.
- **Misleading on imbalanced datasets** — e.g. if 95% of emails aren't spam, a model that always predicts "not spam" scores 95% accuracy while being useless.

### Precision

$$Precision = \frac{TP}{TP + FP}$$

- Of everything I predicted as positive, how many actually were positive?
- Matters when **false positives are costly** — e.g. flagging a legitimate email as spam (user misses important mail).

### Recall (Sensitivity)

$$Recall = \frac{TP}{TP + FN}$$

- Of everything that was actually positive, how many did I correctly catch?
- Matters when **false negatives are costly** — e.g. a medical test missing an actual disease case.

### The Precision-Recall Tradeoff

Raising my classification threshold (e.g. from 0.5 to 0.8) makes the model more conservative about predicting "positive" — Precision goes up (fewer false alarms), but Recall goes down (more real positives get missed). Lowering the threshold does the opposite. There's no universally correct threshold — it depends on which error type is more costly for the specific problem.

### F1-Score

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

- The **harmonic mean** of Precision and Recall (not the simple average) — harmonic mean punishes imbalance between the two more heavily, so F1 is only high when both Precision and Recall are reasonably high.
- Useful as a single summary number when I care about both false positives and false negatives, especially on imbalanced datasets where plain Accuracy is unreliable.

### ROC Curve and AUC

The **ROC (Receiver Operating Characteristic) curve** plots:

$$\text{True Positive Rate (Recall)} \text{ vs. } \text{False Positive Rate} = \frac{FP}{FP + TN}$$

across every possible classification threshold, not just 0.5. It shows the tradeoff visually — how much Recall I gain per unit of false-positive-rate increase as I loosen the threshold.

**AUC (Area Under the Curve)** condenses that curve into a single number between 0 and 1:
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → no better than random guessing
- AUC lets me compare models **independent of any specific threshold choice** — useful when I haven't decided yet what threshold fits the business problem.

---

## Quick summary table

| Metric | Formula | Best used when |
|---|---|---|
| Accuracy | $(TP+TN)/\text{Total}$ | Classes are balanced |
| Precision | $TP/(TP+FP)$ | False positives are costly |
| Recall | $TP/(TP+FN)$ | False negatives are costly |
| F1-Score | Harmonic mean of Precision & Recall | Need balance, imbalanced classes |
| AUC-ROC | Area under TPR vs FPR curve | Comparing models across all thresholds |

---